# Assignment 02: Autograd Exploration (100 points)

**Unit 06: Programming PyTorch | AI 310**

Automatic differentiation is the engine behind neural network training. In this assignment, you will trace computation graphs, compute higher-order derivatives, and implement the derivative computations needed for Physics-Informed Neural Networks (PINNs) — a topic directly tested in USAAIO 2025 Round 2.

**Notation**:
- $\nabla_x f$ = gradient of $f$ with respect to $x$
- $f'(x)$, $f''(x)$ = first and second derivatives
- `create_graph=True` = preserve the computation graph for higher-order derivatives

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries beyond those provided above.

---

## Part 1 (10 points, non-coding)

For $f(x) = 3x^3 - 2x^2 + x - 5$, compute by hand:

1. $f'(x)$
2. $f'(2)$
3. $f''(x)$
4. $f''(2)$

Write your derivations in the cell below.

### WRITE YOUR SOLUTION HERE ###

*Your derivations here*

""" END OF THIS PART """

---

## Part 2 (15 points, coding)

Verify your Part 1 answers using PyTorch autograd.

Compute $f'(2)$ and $f''(2)$ for $f(x) = 3x^3 - 2x^2 + x - 5$ using `torch.autograd.grad` with `create_graph=True`.

Store the results in `f_prime_at_2` and `f_double_prime_at_2`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

x = torch.tensor([2.0], requires_grad=True)

# Compute f(x) = 3x^3 - 2x^2 + x - 5
f = ...

# Compute f'(2) using autograd.grad
f_prime_at_2 = ...  # Should be a scalar tensor

# Compute f''(2) using autograd.grad on the first derivative
f_double_prime_at_2 = ...  # Should be a scalar tensor

In [ ]:
""" END OF THIS PART """
# f'(x) = 9x^2 - 4x + 1, f'(2) = 36 - 8 + 1 = 29
assert torch.allclose(f_prime_at_2, torch.tensor([29.0])), f"Expected 29.0, got {f_prime_at_2.item()}"
# f''(x) = 18x - 4, f''(2) = 36 - 4 = 32
assert torch.allclose(f_double_prime_at_2, torch.tensor([32.0])), f"Expected 32.0, got {f_double_prime_at_2.item()}"
print("Part 2 passed!")

---

## Part 3 (15 points, coding)

**Gradient of a multi-variable function.**

Given $g(x, y) = x^2 y + \sin(xy)$, compute:

1. $\frac{\partial g}{\partial x}$ at $(x, y) = (1, \pi)$. Store as `dg_dx`.
2. $\frac{\partial g}{\partial y}$ at $(x, y) = (1, \pi)$. Store as `dg_dy`.
3. $\frac{\partial^2 g}{\partial x^2}$ at $(x, y) = (1, \pi)$. Store as `d2g_dx2`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

x = torch.tensor([1.0], requires_grad=True)
y = torch.tensor([torch.pi], requires_grad=True)

# g(x, y) = x^2 * y + sin(x * y)
g = ...

# Compute partial derivatives
dg_dx = ...       # partial g / partial x at (1, pi)
dg_dy = ...       # partial g / partial y at (1, pi)
d2g_dx2 = ...     # partial^2 g / partial x^2 at (1, pi)

In [ ]:
""" END OF THIS PART """
# dg/dx = 2xy + y*cos(xy), at (1, pi): 2*pi + pi*cos(pi) = 2*pi - pi = pi
assert torch.allclose(dg_dx, torch.tensor([torch.pi]), atol=1e-5), f"Expected pi, got {dg_dx.item()}"
# dg/dy = x^2 + x*cos(xy), at (1, pi): 1 + cos(pi) = 1 - 1 = 0
assert torch.allclose(dg_dy, torch.tensor([0.0]), atol=1e-5), f"Expected 0, got {dg_dy.item()}"
# d2g/dx2 = 2y - y^2*sin(xy), at (1, pi): 2*pi - pi^2*sin(pi) = 2*pi - 0 = 2*pi
assert torch.allclose(d2g_dx2, torch.tensor([2 * torch.pi]), atol=1e-4), f"Expected 2*pi, got {d2g_dx2.item()}"
print("Part 3 passed!")

---

## Part 4 (20 points, coding)

**PINN-style derivative computation (USAAIO 2025 Round 2 pattern).**

A Physics-Informed Neural Network approximates the solution $u(x)$ to a differential equation. The network takes spatial coordinates as input and outputs the solution value.

Given a simple network `net` that maps $x \to u(x)$, implement a function that computes:
- $u(x)$ — the network output
- $u'(x) = \frac{du}{dx}$ — first derivative
- $u''(x) = \frac{d^2u}{dx^2}$ — second derivative

The function must use `torch.autograd.grad` with `create_graph=True` so that the PDE loss can be backpropagated through.

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
torch.manual_seed(42)

net = nn.Sequential(
    nn.Linear(1, 32),
    nn.Tanh(),
    nn.Linear(32, 32),
    nn.Tanh(),
    nn.Linear(32, 1),
)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_derivatives(net, x):
    """
    Compute u(x), u'(x), and u''(x) where u = net(x).
    
    Args:
        net: nn.Module mapping (N, 1) -> (N, 1)
        x: tensor of shape (N, 1) with requires_grad=True
    
    Returns:
        u: (N, 1) network output
        u_x: (N, 1) first derivative du/dx
        u_xx: (N, 1) second derivative d^2u/dx^2
    """
    # YOUR CODE HERE
    pass

In [ ]:
""" END OF THIS PART """
x_test = torch.linspace(0, 1, 50).reshape(-1, 1)
x_test.requires_grad_(True)
u, u_x, u_xx = compute_derivatives(net, x_test)
assert u.shape == (50, 1), f"u shape: {u.shape}"
assert u_x.shape == (50, 1), f"u_x shape: {u_x.shape}"
assert u_xx.shape == (50, 1), f"u_xx shape: {u_xx.shape}"
# Verify gradients flow: compute a loss and backprop
pde_residual = u_xx + u  # harmonic oscillator: u'' + u = 0
loss = (pde_residual ** 2).mean()
loss.backward()
# Check that network parameters received gradients
has_grads = all(p.grad is not None for p in net.parameters())
assert has_grads, "Gradients did not flow through to network parameters!"
print(f"Part 4 passed! PDE residual loss: {loss.item():.6f}")

---

## Part 5 (20 points, coding)

**Compute partial derivatives for the 2D heat equation.**

The heat equation in 1D: $\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$

Given a network `heat_net` that takes `(x, t)` as input (shape `(N, 2)`) and outputs `u(x, t)` (shape `(N, 1)`), implement a function that computes:
- $u_t = \frac{\partial u}{\partial t}$
- $u_{xx} = \frac{\partial^2 u}{\partial x^2}$
- The PDE residual: $r = u_t - \alpha u_{xx}$

Use $\alpha = 0.01$.

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
torch.manual_seed(42)

heat_net = nn.Sequential(
    nn.Linear(2, 64),
    nn.Tanh(),
    nn.Linear(64, 64),
    nn.Tanh(),
    nn.Linear(64, 1),
)

alpha = 0.01

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def heat_equation_residual(net, x, t, alpha):
    """
    Compute the heat equation PDE residual: u_t - alpha * u_xx.
    
    Args:
        net: nn.Module mapping (N, 2) -> (N, 1)
        x: (N, 1) spatial coordinates, requires_grad=True
        t: (N, 1) temporal coordinates, requires_grad=True
        alpha: thermal diffusivity constant
    
    Returns:
        u: (N, 1) network output
        u_t: (N, 1) time derivative
        u_xx: (N, 1) second spatial derivative
        residual: (N, 1) PDE residual (should be zero if PDE is satisfied)
    """
    # YOUR CODE HERE
    pass

In [ ]:
""" END OF THIS PART """
N = 100
x_pts = torch.rand(N, 1, requires_grad=True)
t_pts = torch.rand(N, 1, requires_grad=True)

u, u_t, u_xx, residual = heat_equation_residual(heat_net, x_pts, t_pts, alpha)

assert u.shape == (N, 1), f"u shape: {u.shape}"
assert u_t.shape == (N, 1), f"u_t shape: {u_t.shape}"
assert u_xx.shape == (N, 1), f"u_xx shape: {u_xx.shape}"
assert residual.shape == (N, 1), f"residual shape: {residual.shape}"

# Verify residual formula
expected_residual = u_t - alpha * u_xx
assert torch.allclose(residual, expected_residual, atol=1e-6)

# Verify gradients flow through
heat_net.zero_grad()
pde_loss = (residual ** 2).mean()
pde_loss.backward()
has_grads = all(p.grad is not None and p.grad.abs().sum() > 0 for p in heat_net.parameters())
assert has_grads, "Gradients must flow to network parameters!"
print(f"Part 5 passed! PDE loss: {pde_loss.item():.6f}")

---

## Part 6 (20 points, coding)

**Train a PINN to solve $u'' + u = 0$ with boundary conditions.**

The simple harmonic oscillator ODE: $u''(x) + u(x) = 0$ on $x \in [0, 2\pi]$.

Boundary conditions: $u(0) = 0$, $u(\pi) = 0$.

The exact solution is $u(x) = A\sin(x)$ for any constant $A$.

Train the network to satisfy both the ODE and the boundary conditions. Your compound loss should be:

$$\mathcal{L} = \mathcal{L}_{\text{ODE}} + \lambda_{\text{BC}} \cdot \mathcal{L}_{\text{BC}}$$

where $\mathcal{L}_{\text{ODE}} = \frac{1}{N}\sum_i (u''(x_i) + u(x_i))^2$ and $\mathcal{L}_{\text{BC}} = u(0)^2 + u(\pi)^2$.

Use $\lambda_{\text{BC}} = 100$ and train for 2000 iterations.

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
torch.manual_seed(42)

pinn = nn.Sequential(
    nn.Linear(1, 64),
    nn.Tanh(),
    nn.Linear(64, 64),
    nn.Tanh(),
    nn.Linear(64, 1),
)

lambda_bc = 100.0
n_collocation = 200
n_iterations = 2000

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# Set up optimizer
# Training loop:
#   1. Sample collocation points in [0, 2*pi]
#   2. Compute u, u_x, u_xx using autograd.grad with create_graph=True
#   3. Compute ODE residual loss: (u_xx + u)^2
#   4. Compute BC loss: u(0)^2 + u(pi)^2
#   5. Total loss = ODE loss + lambda_bc * BC loss
#   6. Backprop and update

# YOUR CODE HERE

In [ ]:
""" END OF THIS PART """
# Test: check boundary conditions
with torch.no_grad():
    u_at_0 = pinn(torch.tensor([[0.0]])).item()
    u_at_pi = pinn(torch.tensor([[torch.pi]])).item()
    print(f"u(0) = {u_at_0:.6f} (should be ~0)")
    print(f"u(pi) = {u_at_pi:.6f} (should be ~0)")

assert abs(u_at_0) < 0.05, f"u(0) = {u_at_0}, should be ~0"
assert abs(u_at_pi) < 0.05, f"u(pi) = {u_at_pi}, should be ~0"

# Test: check ODE residual at interior points
x_check = torch.linspace(0.1, 2*torch.pi - 0.1, 100).reshape(-1, 1)
x_check.requires_grad_(True)
u_check = pinn(x_check)
u_x = torch.autograd.grad(u_check, x_check, torch.ones_like(u_check), create_graph=True)[0]
u_xx = torch.autograd.grad(u_x, x_check, torch.ones_like(u_x))[0]
residual_check = (u_xx + u_check).detach()
mean_residual = (residual_check ** 2).mean().item()
print(f"Mean squared ODE residual: {mean_residual:.6f} (should be < 0.01)")
assert mean_residual < 0.01, f"ODE residual too large: {mean_residual}"
print("Part 6 passed!")